# DINOv2 — Visualisation des features par PCA

Ce notebook utilise le modèle **DINOv2** (Vision Transformer Giant, patch 14) de Facebook Research pour extraire les patch tokens de deux images, puis applique une **PCA** sur les features combinées afin de visualiser les correspondances visuelles entre les deux images.

**Pipeline :**
1. Chargement du modèle DINOv2 pré-entraîné
2. Pré-traitement des images
3. Extraction des patch tokens via `forward_features`
4. PCA sur les tokens des deux images combinées
5. Visualisation interactive avec seuillage sur la première composante principale

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import einops
import torch
import torchvision.transforms as transforms
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from ipywidgets import interact, FloatSlider, Checkbox
# import torch_utils   # utilitaire local (sélection GPU)
import seaborn as sns
import mediapy as mp

## 1. Configuration

In [ ]:
# ── Chemins des images à comparer ────────────────────────────────────────────
img1 = 'E0000014.JPG'
img2 = 'f°93r_dragon.png'


# ── Paramètres PCA / grille de patches ───────────────────────────────────────
# n : nombre de patches par côté  →  image redimensionnée à (n*14, n*14) pixels
# DINOv2 utilise des patches de 14×14 pixels
N_PATCHES = 100          # grille 100×100 patches
N_PCA_COMPONENTS = 25   # nombre de composantes PCA à calculer
IMG_SIZE = N_PATCHES * 14  # = 1400 px

In [ ]:
def which_device() -> str:
    """
    Detects the best available device for PyTorch-based inference (CUDA, MPS, XLA/TPU, or CPU).

    Returns
    -------
    str
        The best available device, one of: 'cuda', 'mps', 'xla', or 'cpu'.
    """

    # 1. CUDA (NVIDIA GPUs)
    if torch.cuda.is_available():
        print(f"✅ Using CUDA GPU: {torch.cuda.get_device_name(0)}")
        return "cuda"

    """# 2. MPS (Apple Silicon GPUs)
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("✅ Using Apple Silicon GPU (MPS)")
        return "mps"
"""
    # 3. TPU (XLA - PyTorch/XLA)
    try:
        import torch_xla.core.xla_model as xm
        dev = xm.xla_device()
        print(f"✅ Using TPU: {dev}")
        return "xla"
    except ImportError:
        pass  # torch_xla not installed or no TPU available

    # 4. CPU fallback
    print("⚠️ Using CPU")
    return "cpu"


In [ ]:
# ── Sélection du device ──────────────────
device = which_device()
print(f"Device utilisé : {device}")

## 2. Chargement du modèle DINOv2

In [ ]:
# ── Chargement de DINOv2 ViT-G/14 depuis torch.hub ───────────────────────────
# ViT-G/14 = Vision Transformer Giant avec patches 14×14
# model.eval() désactive dropout/batchnorm pour l'inférence
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14').to(device)
model = model.eval()

## 3. Pré-traitement des images

In [ ]:
# ── Pipeline de transformation ImageNet ──────────────────────────────────────
# - Resize à IMG_SIZE×IMG_SIZE pour avoir exactement N_PATCHES² patches
# - Normalisation avec les moyennes/écarts-types ImageNet (standard pour DINOv2) pour améliorer la qualité des résultats
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])

In [ ]:
# ── Chargement et conversion en RGB ──────────────────────────────────────────
im1 = Image.open(img1).convert('RGB')
im2 = Image.open(img2).convert('RGB')

print(f"Image 1 : {im1.size}")
print(f"Image 2 : {im2.size}")

In [ ]:
# ── Application du pipeline + envoi sur GPU ───────────────────────────────────
# Ajout d'une dimension batch avec [None] : (C, H, W) → (1, C, H, W)
tim1 = transform(im1).to(device)
tim2 = transform(im2).to(device)

## 4. Extraction des features (patch tokens)

In [ ]:
# ── Inférence sans calcul de gradient (économie mémoire GPU) ─────────────────
# forward_features retourne un dict contenant :
#   - 'x_norm_patchtokens' : tokens des patches normalisés  → (1, N_patches, D)
#   - 'x_norm_clstoken'    : token [CLS] global             → (1, D)
with torch.no_grad():
    ret1 = model.forward_features(tim1[None])
    ret2 = model.forward_features(tim2[None])

# Extraction et aplatissement de la dimension batch : (1, N, D) → (N, D)
tok1 = ret1['x_norm_patchtokens'].squeeze()  # (N_PATCHES², D)
tok2 = ret2['x_norm_patchtokens'].squeeze()  # (N_PATCHES², D)

print(f"Shape tokens image 1 : {tok1.shape}")
print(f"Shape tokens image 2 : {tok2.shape}")

## 5. PCA sur les tokens combinés

In [ ]:
# ── Concaténation des tokens des deux images ──────────────────────────────────
# On empile les deux images pour faire une PCA commune :
# les composantes sont ainsi cohérentes entre les deux images
tok = np.vstack([
    tok1.cpu().numpy(),  # (N_PATCHES², D)
    tok2.cpu().numpy()   # (N_PATCHES², D)
])  # → (2 * N_PATCHES², D)

print(f"Shape matrice combinée : {tok.shape}")

In [ ]:
# ── Ajustement et transformation PCA ─────────────────────────────────────────
pca = PCA(n_components=N_PCA_COMPONENTS)
pim = pca.fit_transform(tok)  # (2 * N_PATCHES², N_PCA_COMPONENTS)

print(f"Shape après PCA : {pim.shape}")

In [ ]:
# ── Variance expliquée par composante ────────────────────────────────────────
plt.figure(figsize=(8, 3))
plt.plot(pca.explained_variance_ratio_, 'k.-')
plt.xlabel('Composante principale')
plt.ylabel('Variance expliquée')
plt.title('Variance expliquée par composante PCA')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Variance cumulée (25 composantes) : {pca.explained_variance_ratio_.sum():.2%}")

## 6. Reshape et visualisation

In [ ]:
# ── Reshape : tokens plats → grille 2D de patches ────────────────────────────
# pim contient les 2 images à la suite : on les sépare en (n_images, H, W, P)
pca_stack = einops.rearrange(
    pim,
    '(n r c) p -> n r c p',
    n=2,
    r=N_PATCHES,
    c=N_PATCHES
)  # (2, N_PATCHES, N_PATCHES, N_PCA_COMPONENTS)

# Juxtaposition horizontale des deux images pour comparaison côte à côte
pair_pca = einops.rearrange(
    pca_stack,
    'n r c p -> r (n c) p'
)  # (N_PATCHES, 2*N_PATCHES, N_PCA_COMPONENTS)

print(f"Shape paire PCA : {pair_pca.shape}")

In [ ]:
# ── Distribution de la 1ère composante principale ────────────────────────────
# PC1 sépare généralement le fond (background) du sujet principal
sns.displot(pim[..., 0].flatten(), kind='kde')
plt.title('Distribution de PC1 (les deux images)')
plt.show()

## 7. Visualisation interactive — Seuillage sur PC1

La 1ère composante principale (PC1) permet de distinguer le fond du premier plan. En appliquant un seuil sur PC1, on peut isoler les régions d'intérêt, puis recalculer une PCA locale uniquement sur ces patches pour obtenir une colorisation RGB significative.

In [ ]:
# ── Première composante principale sur la paire ───────────────────────────────
pair_pca0 = pair_pca[..., 0]  # (N_PATCHES, 2*N_PATCHES)


def apply_threshold(threshold: float, flip: bool):
    """
    Applique un seuil sur PC1 pour sélectionner les patches d'avant-plan,
    recalcule une PCA sur ces patches, normalise en [0,1] puis affiche :
      - l'image originale
      - le masque binaire (patches sélectionnés)
      - les composantes PCA 1-3, 2-4, 3-5 colorisées
      - toutes les composantes PCA individuellement

    Args:
        threshold : valeur de seuil sur PC1
        flip      : si True, sélectionne les patches > threshold (avant-plan)
                    si False, sélectionne les patches < threshold (fond)
    """
    # Création du masque binaire
    good = (pair_pca0 > threshold) if flip else (pair_pca0 < threshold)

    # PCA locale sur les patches sélectionnés uniquement
    good_pca = pca.fit_transform(pair_pca[good, :])

    # Normalisation min-max → valeurs dans [0, 1] pour l'affichage RGB
    good_norm_pca = MinMaxScaler().fit_transform(good_pca)

    """ ICI RECUPERER les good_norm_pca pour voir ce qu'il y a dedans """

    # Image résultat initialisée à zéro (fond noir) + remplissage des patches sélectionnés
    result = np.zeros((N_PATCHES, 2 * N_PATCHES, pca.n_components))
    result[good, :] = good_norm_pca

    # ── Affichage principal : 5 panneaux côte à côte ──────────────────────────
    fig, ax = plt.subplots(1, 5, figsize=(25, 5))

    ax[0].imshow(im1)
    ax[0].set_title('Image originale')
    ax[0].axis('off')

    ax[1].imshow(good, cmap='gray')
    ax[1].set_title(f'Masque (seuil={threshold:.0f})')
    ax[1].axis('off')

    ax[2].imshow(result[..., :3])
    ax[2].set_title('PCA composantes 1-3 (RGB)')
    ax[2].axis('off')

    ax[3].imshow(result[..., 1:4])
    ax[3].set_title('PCA composantes 2-4 (RGB)')
    ax[3].axis('off')

    ax[4].imshow(result[..., 2:5])
    ax[4].set_title('PCA composantes 3-5 (RGB)')
    ax[4].axis('off')

    plt.suptitle('Analyse PCA des features DINOv2', fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── Toutes les composantes PCA individuellement ───────────────────────────
    mp.show_images(
        result.transpose(-1, 0, 1),   # (P, H, W)
        columns=5,
        height=250,
        titles=[f'PC{i+1}' for i in range(pca.n_components)]
    )

In [ ]:
# ── Widget interactif ─────────────────────────────────────────────────────────
# Le slider couvre la plage complète de PC1 ; la checkbox inverse le masque
min_val = int(np.min(pair_pca0))
max_val = int(np.max(pair_pca0))

threshold_slider = FloatSlider(
    min=min_val,
    max=max_val,
    step=1,
    value=0,        # valeur initiale centrée
    description='Seuil PC1',
    style={'description_width': 'initial'}
)
flip_checkbox = Checkbox(
    value=False,
    description='Inverser masque'
)

interact(apply_threshold, threshold=threshold_slider, flip=flip_checkbox)